In [0]:
# Leitura direta das tabelas do schema operacoes.servicos
df_clientes = spark.table("operacoes.servicos.cliente")
df_transacoes = spark.table("operacoes.servicos.transacoes")
df_produtos = spark.table("operacoes.servicos.produtos")
df_cotacao = spark.table("operacoes.servicos.cotacao")

print("Tabelas carregadas com sucesso!")
print(f"Clientes: {df_clientes.count()} registros")
print(f"Transações: {df_transacoes.count()} registros")
print(f"Produtos: {df_produtos.count()} registros")
print(f"Cotações USD: {df_cotacao.count()} registros")

In [0]:
# Join entre Clientes, Transações e Produtos
df_clientes_transacoes = df_transacoes.join(
    df_clientes,
    on="ID_Cliente",
    how="left"
).join(
    df_produtos,
    on="ID_Produto",
    how="left"
)

print(f"Join realizado! Total de registros: {df_clientes_transacoes.count()}")
print("\nPrimeiras linhas:")
display(df_clientes_transacoes.limit(10))

In [0]:
from pyspark.sql.functions import to_date, col

# Converter datas para o formato correto
df_transacoes_date = df_clientes_transacoes.withColumn(
    "Data_Transacao_Formatted", 
    to_date(col("Data_Transacao"), "dd/MM/yyyy")
)

df_cotacao_date = df_cotacao.withColumn(
    "dt_cotacao_formatted",
    to_date(col("dt_cotacao"), "yyyy-MM-dd")
).filter(col("tipo_boletim") == "Fechamento")  # Usar apenas cotação de fechamento

# Join completo: Transações + Clientes + Cotação USD
df_completo = df_transacoes_date.join(
    df_cotacao_date,
    df_transacoes_date["Data_Transacao_Formatted"] == df_cotacao_date["dt_cotacao_formatted"],
    how="left"
).select(
    "ID_Transacao",
    "ID_Cliente",
    "Nome_Cliente",
    "ID_Produto",
    "Nome_Produto",
    col("Valor"),
    df_produtos["Moeda"].alias("Moeda_Produto"),
    "Data_Transacao",
    "Canal_Compra",
    "Data_Criacao",
    col("vl_compra"),
    col("vl_venda"),
    col("dt_cotacao")
)

print(f"Base completa com {df_completo.count()} registros")
print("\nPrimeiras linhas:")
display(df_completo.limit(10))
df_completo.filter("vl_compra is null").count()